<a href="https://colab.research.google.com/github/nick-allen21/Biomedical_Image_Analysis/blob/main/BMI260_Problem_Set_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BIOMEDIN 260/RAD260: Problem Set 3 - Vision Language Foundation Models for Zero-Shot Classification

## **Motivation**:
### A paradigm shift driven by Foundation Models

In recent years large foundation models, pre-trained in an unsupervised manner with massive datasets, have been successfully adapted to wide range of in-domain downstream tasks. Such large models have surpassed both specialized supervised systems and have reached expert level performance in zero and few-shot settings [(Nogare et al., 2023)](https://www.nature.com/articles/s41592-023-01929-5), marking an unprecedented progress in AI (artificial intelligence).

### But what is a Foundation Model?

**Short answer:** Any model that is trained on broad data (generally using self-supervision at scale) that can be adapted to a wide range of downstream tasks [(Bommasani et al., 2021)](https://arxiv.org/pdf/2108.07258)

### What is so special about Foundation Models?
 While foundation models are based on standard deep learning and transfer learning (as any other deep-learning algorithm), their scale results in new emergent capabilities [(Valmeekam et al., 2023)](https://proceedings.neurips.cc/paper_files/paper/2023/file/efb2072a358cefb75886a315a6fcf880-Paper-Conference.pdf)

### Missing the Full picture
Despite the imminent proliferation of foundational models, our comprehension of their operational mechanisms, failure modes, and full potential remains ambiguous owing to their emergent properties. This has sparked new reach direction on how to evaluate Foundation Models in order to holistically capture their performance across different evaluation dimensions. [Liang et al., 2023](https://arxiv.org/pdf/2211.09110)


## Goals:

As stated before, in contrast to specialized  AI models, foundation models can be evaluated across various tasks, often through benchmarks involving multiple datasets. However, this process can be laborious, as datasets may need to be standardized to enable inference with the same infrastructure. In this assignment, the goal is to develop a codebase for efficiently accomplishing this task.

Specifically, you will create an API to facilitate inference with a state-of-the-art biomedical vision-language model called BiomedClip. You will evaluate BiomedClip's zero-shot classification performance across three different modalities. Additionally, you will engage in prompt engineering, modifying the system prompts to **drastically**   enhance performance  (gains larger than 20%) of our zero-shot classifier (**without a single gradient update**).

After getting familiar with BiomedCLIP, we will learn how to use a Multimodal LLM. We will also evaluate this Multimodal LLM on similar tasks and compare performance!

Additionally, you may fine-tune an MLLM (for an extra 25 points!)

By completing this assignment, you will gain familiarity with cutting-edge AI methods. Let's get started!



## Problem 1: Let's meet BiomedCLIP (10 points)

Before we start let's get familiarize (in a conceptual level) with BiomedCLIP.

**Please make sure to input your question in the designated field.**





### **Question 1.1: Describe the BiomedCLIP Modeling in up to three sentences ( 2.5 points)**
(**Hint:** Look at the Modeling section in the original paper)


BiomedCLIP is an adaptation of CLIP tailored to biomedical data, trained solely on PubMed images paired with domain-specific text encoded by PubMedBERT unlike general-domain CLIP (which uses internet data and GPT-2). It uses CLIP’s contrastive learning and also includes enhancements like patch dropout and resolution scaling to improve performance. It's key innovation is domain-specific pretraining from the outset, unlike PubMedCLIP or MedCLIP, which retrofit general CLIP**




[link text](https://)### **Question 1.2: Describe the Dataset used to train BiomedCLIP in up to three sentences 2.5 ( points)**

BiomedCLIP is trained on the PMC-15M dataset, which contains over 15 million figure-caption pairs extracted from more than 4 million full-text  research articles in PubMed. The images and captionss are significantly larger than the standard sizes of 224 X 224 and 77 respectively requiring architectural adjustments. To evaluate the diversity of image types, the authors constructed a taxonomy system which revealed PMC-15M had a wide range of biomedical images including statistical plots, radiology scans, digital pathology, and microscopy.



### **Question 1.3: Describe the Benchmark used to test BiomedCLIP in up to four sentences 2.5 ( points)**

BiomedCLIP is evaluated on a benchmark of eight biomedical vision-language datasets covering image-to-text and text-to-image retrieval, image classification, and visual question answering. These benchmarks were compiled to standardize evaluation across models, since prior work often used different tasks and datasets.


### **Question 1.4: What problems might arise  during testing if we don't have acces to the training dataset? (2.5  points)**

(**Hint:** Imagine edge cases that would make our evaluation flawed )

If we do not have access to the training dataset, we might accidentally include test images that were also used during training, leading to redundancy and inflated performance scores. Additionally, without knowing how the training data was formatted or what metadata was preserved, we risk introducing mismatches or biases that could skew the evaluation results.


Great, now that we know what BiomedCLIP is, let's start coding. We will need to download a couple of dependencies (this should take a at most 2 miniutes)

In [ ]:
!pip install -q scikit-image
!pip install -q open_clip_torch==2.23.0
!pip install -q medmnist

In [ ]:
import medmnist
import torch.utils.data as data
import numpy as np
import matplotlib.pyplot as plt
import random
from   tqdm import tqdm

## Problem 2: Understanig and Parsing our Benchmark Dataset (10 points)


### **Question 2.1: Load Datasets (2.5  points)**

We will benchmark BiomedCLIP using 2D images form MedMNIST [(Yang et. al, 2023)](https://medmnist.com/).  We will start by loading **BloodMNIST** (microscopy),  **DermaMNIST** (dermatology), and  **PneumoniaMNIST** (radiology). make sure to load the **validation** split with image size of **224**




In [ ]:
### (2.5 points)
############################
### Begin your code here ###
############################
from medmnist import BloodMNIST, DermaMNIST, PneumoniaMNIST

image_size:int = 224
split:str = "val"

p_val_dataset = PneumoniaMNIST(split=split, size=image_size, download=True)
b_val_dataset = BloodMNIST(split=split, size=image_size, download=True)
d_val_dataset = DermaMNIST(split=split, size=image_size, download=True)

##########################
### End your code here ###
##########################

########################################
### Do not modify these lines of code ##
########################################
val_datasets = {"PneumoniaMNIST":{"data":p_val_dataset},
                "DermaMNIST":    {"data":d_val_dataset},
                "BloodMNIST":    {"data":b_val_dataset},}
########################################
### Do not modify these lines of code ##
########################################


### **Question 2.2: Add Metadata  (2.5  points)**

If you explore the datasets, you will notice that some tasks involve binary classification, while others involve multi-class classification. It will be important for our pipeline to have these labels available later on. For each dataset in `val_datasets`, add a new key called `task` that indicates the class type. Do the same for the `labels` field. You should accomplish this with a single and simple for-loop.

 (**Hint:** Explore the `medmnist.dataset` class (e.g.`p_val_dataset`), there should be a class attribute that you can access to achieve this)."

In [ ]:
### (2.5 points)
############################
### Begin your code here ###
############################

for key, value in val_datasets.items():
    dataset = value["data"]
    value["labels"] = dataset.labels
    n_classes = len(dataset.info['label'])
    value["task"] = "binary" if n_classes == 2 else "multi-class"
##########################
### End your code here ###
##########################

In [ ]:
val_datasets['PneumoniaMNIST']



 ## 2.3 Create Batch Generator (2.5 points)

We've nearly wrapped up our data functions, but one crucial component is still missing: a data_loader that allows us to iterate over batches. Since the BiomedCLIP preprocessor expects input as a ```PIL.Image object```, we want the data_loader to work directly with the native structure of MedMNIST, where the images are already ```PIL.Image``` objects.



Your task is to implement the function `dataloader`. It should take at least a `dataset` and a `batch_size` and return an iterable object:

Hint: You can either do this by scratch or use ```torch.utils.data.DataLoader```
If you use ```torch.utils.data.DataLoader``` think of what argument you need to add to be able to work with ```PIL.Image```.

Please call your dataloader ```batch_generator```





In [ ]:
from torch.utils.data import DataLoader
batch_size:int=20
num_workers:int=1
dataset_names = list(val_datasets.keys())


for dataset in dataset_names:
  ### (2.5 points)
  ############################
  ### Begin your code here ###
  ############################
    val_datasets[dataset]["batch_generator"] = DataLoader(
        val_datasets[dataset]['data'],
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False,  # Important for validation data
        collate_fn=lambda batch: batch  # Preserve PIL Image objects
    )

    ##########################
    ### End your code here ###
    ##########################

In [ ]:
########################################
### Do not modify these lines of code ##
########################################

########################################
### Do not modify these lines of code ##
########################################
print("Dataset Info:")
print(val_datasets["BloodMNIST"])

print("\nDataset Example:")
# Let's visualize the data: As a started code here is how a single image looks:
batch = next(iter( val_datasets["BloodMNIST"]["batch_generator"]  ))
images, labels = zip(*batch)
labels = np.array(labels).reshape(-1,)
assert len(images) == len(labels)
print(f"Image with label: {labels[-1]}")
images[-1]

########################################
### Do not modify these lines of code ##
########################################

### **Question 2.4: Visualize the dataset  (2.5  points)**


Implement the function `plot_datasets`. This function should take `val_datasets`, iterate over each dataset, and plot `num_examples` images. You can achieve this by either plotting the first `num_examples` or a random collection of `num_examples`. Make sure that each dataset is displayed in a single row (with each example taking a single column).

In [ ]:
def plot_datasets(val_datasets:dict, num_examples:int = 5) -> None:
    """
    Plot examples from validation datasets.

    Args:
    - val_datasets (dict): A dictionary containing validation datasets.
                          Each dataset should have the following structure:
                          {
                              "batch_generator": generator_function,
                              "task": {label: task_name}
                          }
    - num_examples (int): Number of examples to plot from each dataset.
                          Default is 5.
    """

    ### (5 points)
    ############################
    ### Begin your code here ###
    ############################

    fig, axes = plt.subplots(len(val_datasets), num_examples,
                           figsize=(10, 3*len(val_datasets)))

    if len(val_datasets) == 1:
        axes = axes[np.newaxis, :]

    # Iterate through each dataset
    for row_idx, (dataset_name, dataset_info) in enumerate(val_datasets.items()):
        batch = next(iter(dataset_info["batch_generator"]))
        # Randomly select examples if available
        if len(batch) > num_examples:
            batch = random.sample(batch, num_examples)
        # Plot each example
        for col_idx, (image, label) in enumerate(batch):
            ax = axes[row_idx, col_idx]
            ax.imshow(image, cmap='gray' if image.mode == 'L' else None)
            ax.set_title(f"{dataset_name}\nLabel: {label}")
            ax.axis('off')

    plt.tight_layout()
    plt.show()

    ##########################
    ### End your code here ###
    ##########################

plot_datasets(val_datasets, num_examples= 5)

# Problem 3: Model API (15 points)

### **Question 3.1: Create an API for BiomedCLIP (10 points)**

Implement the ```tokenize```, ```preprocess_image```, and ```forward```  methods inside BioMedCLIP. Follow the doc strings for details about the implementation.

(**Hint:** Make sure to consult the official [BioMedCLIP](https://huggingface.co/microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224)  guide for detailed instructions and any specific implementation details. Once you understand the tutorial, implementing these methods should be straightforward.

In [ ]:
import torch
import numpy as np
from PIL import Image
import pathlib
from pathlib import Path
from open_clip import create_model_from_pretrained, get_tokenizer


class BioMedCLIP:
    """
    BioMedClip class for utilizing the BiomedCLIP model.

    Args:
        eval_mode (bool, optional): Whether to set the model in evaluation mode. Defaults to True.
        context_length (int, optional): Length of input context. Defaults to 256.
    """

    def __init__(self,eval_mode:bool=True,context_length:int=256,verbose:bool=True):
        """
        Initialize the Model object.

        Args:
            model: Pre-trained model for image-text matching.
            tokenizer: Tokenizer for processing captions.
            preprocess: Preprocessing function for images.
            context_length: Length of context for tokenization.
            device: Device to run the model on (e.g., "cuda" or "cpu").
        """

        ########################################
        ### Do not modify these lines of code ##
        ########################################
        self.model, self.preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
        self.tokenizer  = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
        self.device     = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
        self.context_length:int = context_length
        self.eval:bool     = eval_mode
        self.verbose:bool  = verbose


        self.model.to(self.device)

        if self.eval:
            self.model.eval()

        if self.verbose:
          print()
          print("="*80)
          print(f"{self.__class__.__name__} model initalized with Eval:{self.eval}, Context Length:{self.context_length}, and device:{self.device}")
          print("="*80)
          print()
        ########################################
        ### Do not modify these lines of code ##
        ########################################




    def tokenize(self,captions:list[str]) -> torch.Tensor:
        """
        Tokenize captions using the biomedclip's tokenizer (self.tokenizer).

        Args:
            captions: Captions to tokenize, either a string or a list of strings.

        Returns:
            Tensor: Tokenized captions.
        """
        ### (2.5 points)
        ############################
        ### Begin your code here ###
        ############################

        tokenized_captions = self.tokenizer(captions, context_length=self.context_length)
        tokenized_captions = tokenized_captions.to(self.device)


        ##########################
        ### End your code here ###
        ##########################

        return tokenized_captions


    def preprocess_image(self,image: str | list[str]) -> torch.Tensor:
        """
        Preprocess images before feeding them into the model.

        Args:
            image: Image file path or a list of image file paths.

        Returns:
            Tensor: Preprocessed images.
        """
        if isinstance(image, str) or isinstance(image, pathlib.Path):
          ### (2.5 points)
          ############################
          ### Begin your code here ###
          ############################

          # Handle both file paths and PIL Images
          if isinstance(image, (str, pathlib.Path)):
              img = Image.open(image).convert('RGB')
          else:
              img = image.convert('RGB')
          preprocessed_image = self.preprocess(img).unsqueeze(0).to(self.device)

          ##########################
          ### End your code here ###
          ##########################


        elif isinstance(image, list) or isinstance(image, tuple):
          ### (2.5 points)
          ############################
          ### Begin your code here ###
          ############################

          # Handle list of either file paths or PIL Images
          processed_images = []
          for img in image:
              if isinstance(img, (str, pathlib.Path)):
                  img = Image.open(img).convert('RGB')
              else:
                  img = img.convert('RGB')
              processed_images.append(self.preprocess(img))
          preprocessed_image = torch.stack(processed_images).to(self.device)

          ##########################
          ### End your code here ###
          ##########################

        return preprocessed_image


    def forward(self,images, texts, return_probs:bool=True) -> dict[str,torch.Tensor]:
        """
        Forward pass through the model.

        Args:
            images: Input images.
            texts: Input texts.
            return_probs: Whether to return class probabilities along with predictions.

        Returns:
            dict: Dictionary containing predictions and optionally class probabilities.


        """
        output:dict     = {}
        output["pred"]  = None

        with torch.no_grad():
          ### (2.5 points)
          ############################
          ### Begin your code here ###
          ############################

            # Preprocess images if they're paths
            if isinstance(images, (str, pathlib.Path, list, tuple)):
                images = self.preprocess_image(images)

            # Tokenize texts if they're strings
            if isinstance(texts, str) or (isinstance(texts, list) and isinstance(texts[0], str)):
                texts = self.tokenize(texts)

            # Get image and text features
            image_features = self.model.encode_image(images)
            text_features = self.model.encode_text(texts)

            # Calculate similarity
            logits_per_image = image_features @ text_features.t()

            output["pred"] = logits_per_image

            if return_probs:
                output["probs"] = logits_per_image.softmax(dim=-1)

          ##########################
          ### End your code here ###
          ##########################



            return output

    ##########################
    ### End your code here ###
    ##########################


In [ ]:
########################################
### Do not modify these lines of code ##
########################################
# Intstanciate a BioMedCLIP object
biomedclip = BioMedCLIP()

# Let's visualize how the data is preprocessed:
# First we iterate over the first batch and get the image and labels
batch = next(iter( val_datasets["PneumoniaMNIST"]["batch_generator"] ))
images, labels = zip(*batch)
labels = np.array(labels).reshape(-1,)
print(f"Image with label: {labels[-1]}")

# Second we preprocess the image:
processed_images = biomedclip.preprocess_image(images)

# Lastly, we plot:
plt.imshow(processed_images[-1].T.cpu())
plt.axis('off')
plt.show()

########################################
### Do not modify these lines of code ##
########################################

### **Question 3.1: Describe the difference between the post-processed and preprocessed images (as in what sort of transformations have been applied to the original image?) (5 points)**
(**Hint:** It might be useful to look under the BiomedCLIP documentation)

Compared to the original images, the preprocessed images used in BiomedCLIP are resized to 224×224 pixels, converted to RGB, normalized, and converted from PIL format (H×W×C) to PyTorch tensor format (C×H×W) to match model input requirements. Post-processed images refer to images that have gone through the model and might be overlaid with prediction heatmaps, segmentation masks, or bounding boxes. So while preprocessed images are standardized for input, post-processed images are often visualizations or annotated versions of the original image that reflect the model’s outputs.




## Problem 4:  Time to Evaluate (20 points)




We will proceed to evaluate each dataset with simple prompts. To achieve this, we will add a new key to each dataset in `val_datasets` called `prompts`.

In [ ]:
########################################
### Do not modify these lines of code ##
########################################

base_prompts:dict[str,str] =  {"PneumoniaMNIST":"chest X-Ray scan displaying  {class}",
                               "DermaMNIST":    "A skin image siplaying   {class}",
                               "BloodMNIST":    "A micrsocopy image of a blood sample dispaiying  {class}",}

for key,value in val_datasets.items():
  value["prompts"]  =  [base_prompts[key].replace("{class}",str(label))  for  label  in value["labels"] ]


print("Example prompt:\n")
print(value["prompts"][0])

########################################
### Do not modify these lines of code ##
########################################



### **Question 4.1: Finalize the inference pipeline (5 points)**

Finish the `get_predictions` function.
1. Get the batch of images and labels.
2. The function should get the correct labels, convert them to a list, and add them to `correct_labels`.
3. Use the BioMedCLIP to preprocess the prompts and images.
4. Do inference using the BioMedCLIP API.
5. Add the predictions to `predictions` (make sure to convert them to a list before appending).

(**Hint:** Use `extend` instead of `append` when adding elements to `correct_labels` and `predictions`.)


In [ ]:
from tqdm import tqdm # Make sure tqdm is imported

def get_predictions(model,val_datasets: dict[str, dict]) -> dict[str, dict]:
    """
    Process data from validation datasets.

    Parameters:
        val_datasets (Dict[str, Dict[str, Any]]): A dictionary containing validation datasets.

    Returns:
        Dict[str, Dict[str, Any]]: A dictionary containing processed data for each dataset.

    Each dataset is processed to generate predictions and correct labels.

    """
    out_dict:dict[str,dict] = {}

    total_datasets = len(val_datasets)
    for key,value in tqdm(val_datasets.items(), total=total_datasets, desc="Processing Datasets"):
      correct_labels:list[int] = []
      predictions:list[int]    = []

      # Make tqdm more robust by adding total and description
      total_batches = len(value['batch_generator'])
      for i, batch in tqdm(enumerate(value['batch_generator']), total=total_batches, desc=f"Processing {key}"):
          # if i ==0:
          #   print("\n\nprompts:")
          #   for prompt_str in value["prompts"]:
          #     print(prompt_str)

          images, labels = zip(*batch)
          correct_labels.extend(np.array(labels).reshape(-1,).tolist())

          # Preprocess images using the model's method
          processed_images = model.preprocess_image(list(images))

          # Tokenize the dataset's prompts using the model's method
          tokenize_prompt  = model.tokenize(value["prompts"])

          # Perform inference using the model's forward method
          # Pass processed images and tokenized prompts
          output           = model.forward(processed_images, tokenize_prompt)

          # Get the predicted class index for each image in the batch
          # The prediction is the index of the text prompt with the highest similarity score
          batch_predictions = output["pred"].argmax(dim=1).tolist()
          predictions.extend(batch_predictions)


      out_dict[key] = {"pred":predictions,"correct":correct_labels,"task":value["task"]}

    return out_dict

**Let's run inference! To efficiently run this part, you will need to switch your instance to GPU (and re-run every cell)**

In [ ]:
########################################
### Do not modify these lines of code ##
########################################
out_dict     = get_predictions(model=biomedclip,val_datasets=val_datasets)   # Infernece
########################################
### Do not modify these lines of code ##
########################################

### **Question 4.2: Finalize the evaluation pipeline (5 points)**

Use ```sklearn.metrics``` to add the specified metrics in ```compute_metrics```


In [ ]:
from sklearn import metrics

def compute_metrics(targets:list[int], predictions:list[int], task_type:str) ->dict[str,float]:

    ### (5 points)
    ############################
    ### Begin your code here ###
    ############################
    metric_results = {}

    # Accuracy
    metric_results['accuracy'] = metrics.accuracy_score(targets, predictions)

    # Balanced Accuracy
    metric_results['balanced_accuracy'] = metrics.balanced_accuracy_score(targets, predictions)

    ### (5 points)
    ############################
    ### Begin your code here ###
    ############################

    # F1 Macro
    metric_results['f1_macro'] = metrics.f1_score(targets, predictions, average='macro')

    # F1 Weighted
    metric_results['f1_weighted'] = metrics.f1_score(targets, predictions, average='weighted')


    if task_type == 'binary': # Corrected from 'binary-class' based on Q2.2 where task is set to 'binary' or 'multi-class'
      # Average Precision - requires probability estimates, but we only have discrete predictions here.
      # For this problem, we'll compute F1 score instead which works with discrete predictions.
      # If probabilities were available, metrics.average_precision_score would be used.

      # F1 Score for binary
      # Set average='binary' for binary classification tasks
      metric_results['f1'] = metrics.f1_score(targets, predictions, average='binary')

    ##########################
    ### End your code here ###
    ##########################

    return metric_results

    ##########################
    ### End your code here ###
    ##########################

**Time to evaluate, run the cell bellow to get your benchmark reuslts:**

In [ ]:
########################################
### Do not modify these lines of code ##
########################################

def compute_and_print_metrics(out_dict: dict[str, dict[str, type]]) -> None:
    """
    Compute metrics for validation datasets and print the results.

    Parameters:
        out_dict (dict): A dictionary containing output predictions and targets where keys are dataset names
                         and values are dictionaries containing predictions and targets.
                         Each dataset dictionary should have keys 'correct' and 'pred', each containing arrays
                         representing correct labels and predicted labels, respectively.

    Returns:
        None

    Example:
        compute_and_print_metrics(out_dict)
    """
    for key,value in val_datasets.items():
      results:dict[str,float] = compute_metrics(targets=out_dict[key]["correct"], predictions=out_dict[key]["pred"],task_type=value['task'])

      print("="*50)
      print(f"Dataset: {key} Task: {value['task']}\nResults:")

      for result_key,result_value in results.items():
        print(f"  {result_key} : {round(result_value,3)}")


compute_and_print_metrics(out_dict)

########################################
### Do not modify these lines of code ##
########################################

### **Question 4.3: Can we outperform BiomedCLIP? (10 points)**

Results don't look so good, right? Well, let's see if we can do better than BiomedCLIP. Familiarize yourself with each dataset and try to classify 15 examples for each dataset (45 images in total). Report your results (using accuracy) here (you are free to write some code to facilitate this task).

**Only report a valid float number (from 0 to 1) with 2 decimal points for each dataset (e.g. 0.56)**


```ANSWER
PneumoniaMNIST: **Your answer goes here**
DermaMNIST:     **Your answer goes here**
BloodMNIST:     **Your answer goes here**
```

This exercise sounds tedious and irrational. However, if we want to creat wonderful prompts we need to understand our dataset. Now that you have somewhat of an idea of how the dataset looks, we are ready to build our own prompts!


As a fun exercise, we will report the human-level performance on these datasets (on ED). To make this evaluation more rigorous, please add your background (e.g. Medical Doctor, CS undergrad, etc.)  here

```ANSWER
Biomedical Computation, Undergrad
```


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

def manual_classify_and_evaluate(val_datasets: dict, num_to_classify: int = 15) -> dict:
    """
    Allows manual classification of a subset of images and calculates accuracy.

    Args:
        val_datasets (dict): Dictionary containing dataset information.
        num_to_classify (int): Number of examples to manually classify from each dataset.

    Returns:
        dict: A dictionary containing the accuracy for each dataset.
    """
    human_accuracies = {}

    for dataset_name, dataset_info in val_datasets.items():
        print(f"\n--- Manual Classification for {dataset_name} ---")
        print(f"Task: {dataset_info['task']}")
        print(f"Labels and descriptions: {dataset_info['data'].info['label']}")

        correct_count = 0
        total_classified = 0
        batch_iterator = iter(dataset_info["batch_generator"])

        # Get enough batches to cover num_to_classify examples
        batches_needed = (num_to_classify + dataset_info['batch_generator'].batch_size - 1) // dataset_info['batch_generator'].batch_size

        examples_to_process = []
        for _ in range(batches_needed):
            try:
                batch = next(batch_iterator)
                examples_to_process.extend(batch)
            except StopIteration:
                break # Not enough data in the generator

        examples_to_process = examples_to_process[:num_to_classify]


        for i, (image, true_label_idx) in enumerate(examples_to_process):
            total_classified += 1
            true_label = dataset_info['data'].info['label'][int(true_label_idx[0])]

            plt.figure(figsize=(3, 3))
            plt.imshow(image, cmap='gray' if image.mode == 'L' else None)
            plt.title(f"Image {i+1}/{num_to_classify} from {dataset_name}")
            plt.axis('off')
            plt.show()

            # Prompt the user for classification
            user_input = input(f"Classify the image (Enter label based on descriptions above, or index {list(dataset_info['data'].info['label'].keys())}): ")

            # Try to match user input to label index or description
            predicted_label_idx = None
            try:
                # Check if input is an integer index
                user_input_int = int(user_input)
                if user_input_int in dataset_info['data'].info['label']:
                    predicted_label_idx = user_input_int
            except ValueError:
                # Check if input matches a label description
                for label_idx, label_desc in dataset_info['data'].info['label'].items():
                    if user_input.lower() == label_desc.lower():
                        predicted_label_idx = label_idx
                        break

            if predicted_label_idx is not None and predicted_label_idx == int(true_label_idx[0]):
                print("Correct!")
                correct_count += 1
            elif predicted_label_idx is not None:
                print(f"Incorrect. True label was: {true_label} (Index: {int(true_label_idx[0])})")
            else:
                print(f"Invalid input. Skipping this image for accuracy calculation. True label was: {true_label} (Index: {int(true_label_idx[0])})")
                total_classified -= 1 # Don't count this image if input was invalid

            print("-" * 20) # Separator

        if total_classified > 0:
            accuracy = correct_count / total_classified
            human_accuracies[dataset_name] = round(accuracy, 2)
            print(f"Accuracy for {dataset_name}: {human_accuracies[dataset_name]:.2f} ({correct_count}/{total_classified})")
        else:
             human_accuracies[dataset_name] = 0.0
             print(f"No valid classifications recorded for {dataset_name}.")

    return human_accuracies

# Run the manual classification and get results
human_benchmark_results = manual_classify_and_evaluate(val_datasets, num_to_classify=15)

# Print the final results in the desired format
print("\n--- Human Benchmark Results ---")
print(f"PneumoniaMNIST: {human_benchmark_results.get('PneumoniaMNIST', 'N/A'):.2f}")
print(f"DermaMNIST:     {human_benchmark_results.get('DermaMNIST', 'N/A'):.2f}")
print(f"BloodMNIST:     {human_benchmark_results.get('BloodMNIST', 'N/A'):.2f}")

## Problem 5: Prompt-engineering (10 points)

### What is prompt-engineering ?

[Prompt engineering](https://aws.amazon.com/what-is/prompt-engineering/)
 involves crafting precise instructions to guide generative AI systems in producing desired outputs. Despite their mimicry of human behavior, generative AI still needs specific guidance to generate high-quality and relevant content. Prompt engineers select suitable formats, phrases, words, and symbols to enhance AI interactions with users, employing creativity and trial and error to refine input texts. Ultimately, their goal is to ensure that the AI functions as intended within applications.


 In this step we will manually define promtps to achive better results:





In [ ]:
# Example of how to add custom prompts per dataset
# Take into consideration that the order of the prompts does matter.
# Prompts should be ordered according to the label order.
# For example, for the PneumoniaMNIST, we first declare the prompt for 0 (normal)
# and then we define the prompt for 1 (pneumonia)

print("Labels:")
print(val_datasets["PneumoniaMNIST"]["labels"])
val_datasets["PneumoniaMNIST"]["prompts"] = ["Chest X-Ray scan displaying healthy lung  without any abnormalities",
                                             "Chest X-Rays can displaying signs of infection such as consolidation or effusion (depicting typical pneumonia symptoms )"]

# DermaMNIST – dermoscopic skin images
derma_label_dict = val_datasets["DermaMNIST"]["data"].info["label"]  # {idx: label_name}
val_datasets["DermaMNIST"]["prompts"] = [
    f"Dermoscopic image showing {label_name}" for label_name in derma_label_dict.values()
]

# BloodMNIST – microscopy of peripheral-blood smears
blood_label_dict = val_datasets["BloodMNIST"]["data"].info["label"]
val_datasets["BloodMNIST"]["prompts"] = [
    f"Light-microscope image of a blood smear with predominantly {label_name}" for label_name in blood_label_dict.values()
]

# Display to verify
print("\nDermaMNIST prompts:")
print(val_datasets["DermaMNIST"]["prompts"][:min(5, len(val_datasets["DermaMNIST"]["prompts"]))])

print("\nBloodMNIST prompts:")
print(val_datasets["BloodMNIST"]["prompts"][:min(5, len(val_datasets["BloodMNIST"]["prompts"]))])

print("\nCustom Prompts:")
print(val_datasets["PneumoniaMNIST"]["prompts"])

### **Question 5.1: Define your own prompts (5 points)**

Define your own prompts to achieve better accuracy results for each dataset.

Your prompt should achieve at least:

- 65% accuracy for PneumoniaMNIST
- 35% accuracy for DermaMNIST
- 30% accuracy for BloodMNIST


**Top 5 prompt submissions will get an additional 10 points for this assignment**


In [ ]:
########################################
### Do not modify these lines of code ##
########################################

out_dict     = get_predictions(model=biomedclip,val_datasets=val_datasets)  # Infernece
compute_and_print_metrics(out_dict)

########################################
### Do not modify these lines of code ##
########################################


### **Question 5.2: Based on your best prompts, what are the most important strategies you had to follow to increase performance? Why do you think that is the case? (5 points)**

Effective prompt engineering for zero-shot image classification with models like BiomedCLIP involves crafting descriptions that are highly specific and clear about the visual features of each class. Utilizing terminology relevant to the medical domain is paramount, as these models are often trained on domain-specific data. Matching the style and content of prompts to the characteristics of the model's training data, such as PubMed image captions, can further enhance performance. For multi-class tasks, ensuring distinct and descriptive prompts for each category, ordered correctly according to labels, is key to helping the model differentiate between classes. Ultimately, successful prompt engineering relies on understanding the dataset and iteratively refining prompts to improve the model's ability to accurately match visual features with textual descriptions.

# Problem 6: Multimodal LLMs for Zero-Shot Classification (35 points)


Problem 6.1: What are Multimodal LLMs? (5 points)

In the previous exercise, we learned how to use an embedding vision-language encoder. Now, we will learn how to use a Multimodal LLM. Naturally, the Multimodal LLM leverages the vision-language encoder we just used. Please provide a brief explanation of how this vision encoder is integrated into an LLM.

Hint: Please read:  [Visual Instruction Tuning](https://arxiv.org/pdf/2304.08485)

Multimodal LLMs are large-language models that have been extended to accept and reason over non-text modalities (images, audio, video, etc.).  A frozen or lightly-trained vision encoder (e.g. a ViT from CLIP/BiomedCLIP) first converts the input image into a sequence of dense embeddings.  A small projection layer then maps these visual embeddings to the same dimensionality as the language tokens and prepends (or interleaves) them with the text tokens that the LLM already understands.  During multimodal instruction tuning the language model learns to attend to the visual tokens, enabling it to generate coherent text that is grounded in the image content.


Great! Now we will do inference with a small 2B multimodal LLM called SmolVLM. Here is the paper: [SmolVLM](https://arxiv.org/abs/2504.05299)


Problem 6.2: Please provide a TL;DR of the main findings from the SmolVLM paper. How do they enable faster inference? (5 points)

**SmolVLM TL;DR** – The authors show that a *very small* 2-billion-parameter decoder-only LLM can reach competitive multimodal performance when paired with a strong frozen vision encoder and instruction-tuned on only a few million image–text pairs.  Key techniques that enable *fast* inference:
• **Tiny decoder** – a 2 B parameter LLM is 5–10 × smaller than many contemporaries, so memory footprint and latency drop proportionally.
• **Weight-only 8-bit quantisation** – the model ships in INT8 which further halves memory and bandwidth requirements with minimal quality loss.
• **Flash-Attention 2 & grouped-query attention** – efficient kernels keep the compute-bound portions of generation fast.
• **Single pooled image token** – instead of feeding hundreds of patch tokens, they project the pooled image representation, greatly shortening the prompt length.
Together, these changes allow SmolVLM to serve interactive multimodal queries on a single consumer-grade GPU in real-time.

Time to code!!!

**Quick Note:** Solving the steps below is trivial with ChatGPT. However, we recommend not using it, as this exercise is meant to get you up to speed with how to use multimodal LLMs with Hugging Face. It’s really important to understand this API if you're interested in MLLM research.

In [ ]:
del biomedclip # Need to del biomedclip to prevent running out of memory

from transformers import AutoProcessor, AutoModelForVision2Seq
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-Instruct")
model = AutoModelForVision2Seq.from_pretrained("HuggingFaceTB/SmolVLM-Instruct",
                                                torch_dtype=torch.bfloat16,).to(DEVICE)


Problem 6.3:^ The implementation above to load the SmolVLM is slow. Try to remember ways we can make a model faster and enable the corresponding flags. (5 points)

 Hint: Lecture 8 might be useful

 Hint: You may need to install additional libraries.

Problem 6.4: Below is a template for performing inference with SmolVLM. Please refer to the documentation to complete it.  (5 points)

We'll use only PneumoniaMNIST, as this model is larger than BiomedCLIP.



In [ ]:


def run_inference(dataset, message_template, prompts, processor, model, device="cuda"):
    """
    Run inference on a dataset using a message template and a vision-language model.

    Args:
        dataset (dict): A dictionary containing 'prompts' and 'batch_generator'.
        message_template (list): A list representing the message structure with placeholders.
        processor (PreTrainedProcessor): The processor that formats input for the model.
        model (PreTrainedModel): The vision-language model for inference.
        device (str): The device to run inference on. Default is "cuda".

    Returns:
        results (list): A list of decoded outputs from the model.
        corrext_answers (list): A list of the gt anwers.
    """
    results = []
    correct_answers = []
    for batch in dataset["batch_generator"]:
        batch_len = len(batch)

        # Prepare messages for each item in the batch
        messages = [message_template for _ in range(batch_len)]

        # Prepare images
        images = [[image] for image, _ in batch]

        # Save Correct answers
        correct_answers.extend([prompts[int(label_index[0])] for _, label_index in batch])

        ############################
        ### Begin your code here ###
        ############################
        # The SmolVLM `processor` expects *lists* of images and paired text.
        # We therefore repeat the same textual message (which already embeds
        # the answer options) once for every image in the current batch.

        batch_messages = [message_template for _ in range(batch_len)]  # deep copy not required for processor

        # Build processor inputs
        inputs = processor(images=[img[0] for img in images],         # images as list of PIL.Image
                           text=batch_messages,                       # conversational template
                           return_tensors="pt").to(device)

        # Generate model outputs (max 20 new tokens keeps generation fast)
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=20)

        # Decode to plain text
        generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

        ##########################
        ### End your code here ###
        ##########################

        results.extend(generated_texts)
        break  # Remove this line if you want to process all batches

    return results,correct_answers

It's time to define our dataloader, 'PneumoniaMNIST', and prompts in order to test our inference code!



In [ ]:
PneumoniaMNIST = val_datasets['PneumoniaMNIST']
prompts = PneumoniaMNIST['prompts']

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": f"Select the correct option that best describes the image? options {PneumoniaMNIST['prompts']}"}
        ]
    },
]

In [ ]:
########################################
### Do not modify these lines of code ##
########################################

results,correct_answers = run_inference(dataset =PneumoniaMNIST,
              message_template = messages,
              prompts = prompts,
              processor = processor,
              model = model)

########################################
### Do not modify these lines of code ##
########################################

Notice that the results contain both the user query and the assistant's response. Remove the user query from the results, and proceed to calculate accuracy using exact match between the results and the correct answers. Please try both the initial baseline prompt and the best prompt you have obtained from BiomedCLIP.

In [ ]:
  ############################
  ### Begin your code here ###
  ############################
  # (10 points)


# First, strip the user part (everything before the string "Assistant:") if it
# exists.  We also normalise capitalisation and white-space so that an exact
# match is still robust to trivial formatting differences.

def _clean_pred(txt: str) -> str:
    if "Assistant:" in txt:
        txt = txt.split("Assistant:")[-1]
    return txt.strip().lower()

clean_preds   = [_clean_pred(t) for t in results]
clean_targets = [t.strip().lower() for t in correct_answers]

# Compute exact-match accuracy
correct = sum(int(p == t) for p, t in zip(clean_preds, clean_targets))
acc     = correct / len(clean_targets) if clean_targets else 0.0

print(f"\nExact-match accuracy for current prompt: {acc:.3f} ({correct}/{len(clean_targets)})")


  ##########################
  ### End your code here ###
  ##########################

Report your results here:



```ANSWER
Acc:
SmolVLM PneumoniaMNIST (baseline prompt): **Your answer goes here**
SmolVLM  PneumoniaMNIST (best biomedclip prompt): **Your answer goes here**

biomedclip PneumoniaMNIST (baseline prompt): **Your answer goes here**
biomedclip  PneumoniaMNIST (best biomedclip prompt): **Your answer goes here**
```




### Extra Points: (25 points)

We have learned how to perform inference with VLMs! If you would like to learn more, let's proceed to fine-tune SmolVLM. Please read the SmolVLM documentation on Hugging Face and fine-tune it on the first 100 examples from the PneumoniaMNIST training set. Does the performance improve?

Note: You need to find the documentation! :)

```ANSWER
SFT Acc:
```

In [ ]:
############################
### Begin your code here ###
############################
# (25 Additional points)



##########################
### End your code here ###
##########################

```
  / __|/ _ \ | \| | / __|| _ \  /_\|_   _|/ __|
 | (__| (_) || .` || (_ ||   / / _ \ | |  \__ \
  \___|\___/ |_|\_| \___||_|_\/_/ \_\|_|  |___

  ```

You're officially done with all assignments for BMI260!


